# Exercise 01: Cloud Deployment
## AIAT 125 - AI Model Deployment

---

## Learning Objectives

In this exercise, you will:
- Deploy models on cloud platforms
- Use managed ML services
- Optimize deployment costs
- Monitor cloud deployments

---

## Real-World Context

You are deploying an ML model to production using cloud ML services (AWS SageMaker, GCP Vertex AI, or Azure ML).

---


Setup
TODO: Install required libraries
%pip install ...

TODO: Import libraries
YOUR CODE HERE

print('✅ Setup complete!')

## Task 1: Cloud Platform Setup (30 points)

Set up deployment on a cloud ML platform (AWS/GCP/Azure).

**Requirements:**
- Choose platform
- Set up account/resources
- Prepare model
- Configure environment


TODO: Implement solution
YOUR CODE HERE


## Task 2: Model Deployment (40 points)

Deploy model using cloud ML services.

**Requirements:**
- Upload model
- Create endpoint
- Test inference
- Monitor deployment


TODO: Implement solution
YOUR CODE HERE


## Task 3: Cost Optimization (30 points)

Optimize deployment costs and resource usage.

**Requirements:**
- Analyze costs
- Optimize instance types
- Implement auto-scaling
- Set up cost alerts


TODO: Implement solution
YOUR CODE HERE



---

## Submission

**Submission**: Complete notebook with all tasks implemented

**Grading**: 100 points total


## 🌍 Real-World Worked Example — Deploy a Trained Model as a REST API

**Industry context:**
- Spotify's recommendation model is served via a FastAPI microservice handling 400M users
- Instagram's image moderation runs as a containerised PyTorch model behind a REST endpoint
- Every ML feature in a modern app goes through a model serving layer like this

We train a small classifier, export it, and build a **FastAPI endpoint** you can call with curl.

In [ ]:
# ── Part 1: Train and save a model ────────────────────────────────────────
import torch, torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

iris = load_iris()
X = StandardScaler().fit_transform(iris.data.astype(np.float32))
y = iris.target
X_tr,X_te,y_tr,y_te = train_test_split(X, y, test_size=0.2, random_state=42)

model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
opt   = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
Xt = torch.tensor(X_tr); Yt = torch.tensor(y_tr, dtype=torch.long)

for _ in range(200):
    loss = loss_fn(model(Xt), Yt)
    opt.zero_grad(); loss.backward(); opt.step()

torch.save(model.state_dict(), '/tmp/iris_model.pt')
print("Model saved to /tmp/iris_model.pt")

# Verify
model.eval()
with torch.no_grad():
    acc = (model(torch.tensor(X_te)).argmax(1)==torch.tensor(y_te)).float().mean()
print(f"Test accuracy: {acc:.2%}")

# ── Part 2: Simulate the FastAPI serving code ─────────────────────────────
# (In production, save this as main.py and run: uvicorn main:app --reload)
fastapi_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import torch, torch.nn as nn
import numpy as np

app = FastAPI(title="Iris Classifier API")

# Load model at startup
model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
model.load_state_dict(torch.load("/tmp/iris_model.pt"))
model.eval()
CLASSES = ["setosa", "versicolor", "virginica"]

class IrisRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.post("/predict")
def predict(req: IrisRequest):
    features = torch.tensor([[req.sepal_length, req.sepal_width,
                               req.petal_length, req.petal_width]])
    with torch.no_grad():
        logits = model(features)
        probs  = torch.softmax(logits, dim=1)[0]
        label  = CLASSES[probs.argmax().item()]
    return {"prediction": label, "confidence": round(probs.max().item(), 3)}

@app.get("/health")
def health(): return {"status": "ok"}

# Run with: uvicorn main:app --host 0.0.0.0 --port 8000
# Test with: curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" \
#            -d '{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'
'''
print("\n── FastAPI serving code (save as main.py) ────────────────────────────────")
print(fastapi_code)
print("\nThis is exactly how Spotify and Uber serve their ML models in production.")

## 📝 Summary

In this notebook, you practiced:
- Deploying an ML model to a **cloud platform** (AWS SageMaker, GCP Vertex AI, or Azure ML)
- Using **managed ML services** to handle infrastructure, scaling, and model versioning
- Configuring **cost optimization** strategies: instance types, auto-scaling, and spot instances
- Setting up **cloud monitoring**: endpoint health checks, latency metrics, and alerting thresholds

**Next steps:** Combine cloud deployment with Docker containerization (Unit 4) for maximum portability, and set up a CI/CD pipeline to automate deployments on every model update.

## 📚 References & Further Reading

**Frameworks:**
- [FastAPI Documentation](https://fastapi.tiangolo.com/) — Modern Python API framework
- [ONNX Runtime](https://onnxruntime.ai/) — Cross-platform inference
- [BentoML](https://github.com/bentoml/BentoML) — ML model serving framework

**Cloud Services:**
- [AWS SageMaker Inference](https://docs.aws.amazon.com/sagemaker/latest/dg/deploy-model.html)
- [Google Cloud Vertex AI](https://cloud.google.com/vertex-ai/docs/predictions/overview)

**State-of-the-Art:** Uber, Airbnb, and Spotify deploy hundreds of ML models using microservices with FastAPI/gRPC.